# [핸즈온 머신러닝] Chapter 1 – 머신러닝 한눈에 보기 (The Machine Learning Landscape)

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/jsmak/handson-mlp-study/blob/main/ch01/01_the_machine_learning_landscape_kr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>
<br clear="left"/>

> **📚 출처 및 라이선스**
>
> - **원본 노트북**: [rickiepark/handson-mlp](https://github.com/rickiepark/handson-mlp) – [`01_the_machine_learning_landscape.ipynb`](https://github.com/rickiepark/handson-mlp/blob/main/01_the_machine_learning_landscape.ipynb) (박해선 님, 『핸즈온 머신러닝 with 사이킷런, 파이토치』 한빛미디어, 2026 예제 코드)
> - **원저작**: [ageron/handson-mlp](https://github.com/ageron/handson-mlp) (Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*, O'Reilly)
> - **라이선스**: 원본 코드는 [Apache License 2.0](https://github.com/rickiepark/handson-mlp/blob/main/LICENSE)을 따릅니다.
> - **변경 사항**: 개인 학습(인프런 챌린지 복습) 목적으로 원본 노트북을 바탕으로 한국어 설명·주석·요약을 추가하고 셀 구성을 일부 재정리했습니다. 책 본문은 포함하지 않습니다.

이 노트북은 1장의 코드 예제와 연습문제 해답을 담고 있습니다. 본격적인 실습은 2장부터이고, 1장은 **"모델이 데이터에서 배운다는 게 무슨 뜻인지"** 감을 잡는 장입니다.

# 1. Setup (환경 설정)

Python 3.10 이상, Scikit-Learn 1.6.1 이상이 필요합니다.

In [ ]:
import sys

# Python 버전 확인 (3.10 이상)
assert sys.version_info >= (3, 10)

In [ ]:
from packaging.version import Version
import sklearn

# Scikit-Learn 버전 확인 (1.6.1 이상)
assert Version(sklearn.__version__) >= Version("1.6.1")

In [ ]:
import matplotlib.pyplot as plt

# 그래프 기본 글꼴 크기 설정
plt.rc('font', size=12)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=12)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

# 2. Code example 1-1 : 돈이 많으면 더 행복할까?

## 2.1 문제 설정

- **입력(x)**: 1인당 GDP (USD)
- **출력(y)**: 삶의 만족도 (Life satisfaction, 0~10점)
- **질문**: 푸에르토리코(1인당 GDP 약 33,443달러)의 삶의 만족도는 얼마일까?

산점도를 그려 보면 점들이 **대략 오른쪽 위로 올라가는 직선** 모양입니다. 그래서 선형 모델을 선택합니다.

## 2.2 선형 회귀 모델

$$
\text{life\_satisfaction} = \theta_0 + \theta_1 \times \text{GDP\_per\_capita}
$$

- $\theta_0$ (절편): GDP가 0일 때의 기본 만족도 → "출발점"
- $\theta_1$ (기울기): GDP가 1달러 늘 때 만족도가 얼마나 오르는지 → "돈의 행복 환율"

모델 학습은 **점들 사이를 가장 잘 지나가는 자(ruler)의 위치와 기울기를 찾는 일**입니다. 자를 결정하는 두 숫자 $\theta_0, \theta_1$을 **모델 파라미터**라고 부릅니다.

> 💡 `X`를 `lifesat[[...]]`처럼 **대괄호 두 개**로 뽑는 이유: Scikit-Learn은 입력을 항상 `(샘플 수, 특성 수)` 형태의 2차원 배열로 받기 때문입니다.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

# 1) 데이터 다운로드 및 준비
data_root = "https://github.com/ageron/data/raw/main/"
lifesat = pd.read_csv(data_root + "lifesat/lifesat.csv")
X = lifesat[["GDP per capita (USD)"]].values   # X: (N, 1)
y = lifesat[["Life satisfaction"]].values      # y: (N, 1)

# 2) 데이터 시각화
lifesat.plot(kind='scatter', grid=True,
             x="GDP per capita (USD)", y="Life satisfaction")
plt.axis([23_500, 62_500, 4, 9])
plt.show()

# 3) 선형 모델 선택
model = LinearRegression()

# 4) 학습: 최소제곱법으로 θ0, θ1 계산
model.fit(X, y)

# 5) 푸에르토리코 예측
X_new = [[33_442.8]]          # (1, 1)
print(model.predict(X_new))   # [[6.01610329]]

## 2.3 손으로 따라가 보는 예측

학습된 파라미터는 $\theta_0 = 3.75$, $\theta_1 = 6.78 \times 10^{-5}$ 입니다. 푸에르토리코 값을 대입하면:

$$
3.75 + 6.78 \times 10^{-5} \times 33{,}442.8 = 3.75 + 2.27 \approx 6.02
$$

`model.predict()`가 하는 일은 결국 곱셈 한 번과 덧셈 한 번입니다. 아래 셀에서 직접 확인해 봅니다.

In [ ]:
# 학습된 파라미터 꺼내기
theta0 = model.intercept_[0]   # 절편
theta1 = model.coef_[0][0]     # 기울기
print(f"θ0={theta0:.2f}, θ1={theta1:.2e}")

# 식에 직접 대입한 결과와 predict 결과 비교
print(theta0 + theta1 * 33_442.8)

## 2.4 k-최근접 이웃(k-NN) 회귀로 바꿔 보기

모델만 바꾸면 전혀 다른 방식의 학습이 됩니다. 코드 두 줄만 교체합니다.

$$
\hat{y} = \frac{1}{k}\sum_{i \in \text{이웃}} y_i
$$

→ "나랑 소득 수준이 비슷한 이웃 나라 3곳은 평균적으로 얼마나 행복한가?"를 재는 것입니다.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

# k=3: GDP가 가장 비슷한 나라 3개를 찾아 만족도를 평균
model = KNeighborsRegressor(n_neighbors=3)

model.fit(X, y)              # 학습 = 데이터를 그대로 저장
print(model.predict(X_new))  # [[5.73333333]]

In [ ]:
# 실제로 어떤 이웃 3개가 선택됐는지 확인
dist, idx = model.kneighbors(X_new)   # dist, idx: (1, 3)
lifesat.iloc[idx[0]]

| 구분 | 선형 회귀 | k-NN 회귀 |
|---|---|---|
| 학습 방식 | **모델 기반** (파라미터를 찾음) | **사례 기반** (데이터를 외움) |
| 예측 방식 | 식에 대입 | 가까운 이웃의 평균 |
| 푸에르토리코 예측 | 6.02 | 5.73 |

# 3. 데이터 생성 및 그림 그리기

원문에서 "건너뛰어도 된다"고 한 부분이지만, 1장의 핵심 개념(샘플링 편향·과대적합·규제)을 보여주는 그림들이 여기서 나옵니다.

## 3.1 삶의 만족도 데이터 불러오기

OECD BLI 데이터(삶의 만족도)와 World Bank 1인당 GDP 데이터(2020년)를 내려받습니다.

In [ ]:
from pathlib import Path
import urllib.request

# 저장 폴더 생성
datapath = Path() / "datasets" / "lifesat"
datapath.mkdir(parents=True, exist_ok=True)

# 파일이 없을 때만 다운로드
data_root = "https://github.com/ageron/data/raw/main/"
for filename in ("oecd_bli.csv", "gdp_per_capita.csv"):
    if not (datapath / filename).is_file():
        print("Downloading", filename)
        url = data_root + "lifesat/" + filename
        urllib.request.urlretrieve(url, datapath / filename)

In [ ]:
oecd_bli = pd.read_csv(datapath / "oecd_bli.csv")
gdp_per_capita = pd.read_csv(datapath / "gdp_per_capita.csv")

GDP 데이터는 2020년만 남기고, 나라 이름을 인덱스로 설정합니다.

In [ ]:
gdp_year = 2020
gdppc_col = "GDP per capita (USD)"
lifesat_col = "Life satisfaction"

gdp_per_capita = gdp_per_capita[gdp_per_capita["Year"] == gdp_year]  # 2020년만
gdp_per_capita = gdp_per_capita.drop(["Code", "Year"], axis=1)       # 불필요한 열 제거
gdp_per_capita.columns = ["Country", gdppc_col]
gdp_per_capita.set_index("Country", inplace=True)

gdp_per_capita.head()

OECD 데이터는 전체 인구 기준(`TOT`)만 남기고 `pivot`으로 '나라 × 지표' 표로 바꿉니다.

In [ ]:
oecd_bli = oecd_bli[oecd_bli["INEQUALITY"] == "TOT"]
oecd_bli = oecd_bli.pivot(index="Country", columns="Indicator", values="Value")

oecd_bli.head()

두 표를 나라 이름 기준으로 합치고 GDP·삶의 만족도 두 열만 남깁니다.

In [ ]:
full_country_stats = pd.merge(left=oecd_bli, right=gdp_per_capita,
                              left_index=True, right_index=True)
full_country_stats.sort_values(by=gdppc_col, inplace=True)
full_country_stats = full_country_stats[[gdppc_col, lifesat_col]]

full_country_stats.head()

> ⚠️ **이 장의 숨은 장치**: 과대적합과 샘플링 편향을 보여주기 위해 GDP 23,500~62,500달러 구간의 나라만 **의도적으로** 골라 씁니다. 나중에 빠진 나라들을 다시 보여주면 같은 직선 경향을 따르지 않는다는 것이 드러납니다.

In [ ]:
min_gdp = 23_500
max_gdp = 62_500

country_stats = full_country_stats[(full_country_stats[gdppc_col] >= min_gdp) &
                                   (full_country_stats[gdppc_col] <= max_gdp)]
country_stats.head()

In [ ]:
# 부분 데이터 / 전체 데이터 저장
country_stats.to_csv(datapath / "lifesat.csv")
full_country_stats.to_csv(datapath / "lifesat_full.csv")

## 3.2 주요 나라 표시하기

In [ ]:
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

min_life_sat = 4
max_life_sat = 9

# 나라 이름 라벨을 둘 위치
position_text = {
    "Turkey": (29_500, 4.2),
    "Hungary": (28_000, 6.9),
    "France": (40_000, 5),
    "New Zealand": (28_000, 8.2),
    "Australia": (50_000, 5.5),
    "United States": (59_000, 5.3),
    "Denmark": (46_000, 8.5)
}

for country, pos_text in position_text.items():
    pos_data_x = country_stats[gdppc_col].loc[country]
    pos_data_y = country_stats[lifesat_col].loc[country]
    country = "US" if country == "United States" else country
    plt.annotate(country, xy=(pos_data_x, pos_data_y),
                 xytext=pos_text, fontsize=12,
                 arrowprops=dict(facecolor='black', width=0.5,
                                 shrink=0.08, headwidth=5))
    plt.plot(pos_data_x, pos_data_y, "ro")   # 빨간 점으로 강조

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()

In [ ]:
highlighted_countries = country_stats.loc[list(position_text.keys())]
highlighted_countries[[gdppc_col, lifesat_col]].sort_values(by=gdppc_col)

## 3.3 파라미터에 따라 달라지는 직선

$\theta_0, \theta_1$ 값을 바꾸면 직선이 완전히 달라집니다. 학습이란 이 중에서 **데이터에 가장 잘 맞는 θ를 찾는 것**입니다.

In [ ]:
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

X = np.linspace(min_gdp, max_gdp, 1000)   # X: (1000,)

# (θ0, θ1) 세 가지 조합을 그려 비교
w1, w2 = 4.2, 0
plt.plot(X, w1 + w2 * 1e-5 * X, "r")
plt.text(40_000, 4.9, fr"$\theta_0 = {w1}$", color="r")
plt.text(40_000, 4.4, fr"$\theta_1 = {w2}$", color="r")

w1, w2 = 10, -9
plt.plot(X, w1 + w2 * 1e-5 * X, "g")
plt.text(26_000, 8.5, fr"$\theta_0 = {w1}$", color="g")
plt.text(26_000, 8.0, fr"$\theta_1 = {w2} \times 10^{{-5}}$", color="g")

w1, w2 = 3, 8
plt.plot(X, w1 + w2 * 1e-5 * X, "b")
plt.text(48_000, 8.5, fr"$\theta_0 = {w1}$", color="b")
plt.text(48_000, 8.0, fr"$\theta_1 = {w2} \times 10^{{-5}}$", color="b")

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()

## 3.4 최적 직선 찾기와 예측

In [ ]:
from sklearn import linear_model

X_sample = country_stats[[gdppc_col]].values   # (N, 1)
y_sample = country_stats[[lifesat_col]].values # (N, 1)

lin1 = linear_model.LinearRegression()
lin1.fit(X_sample, y_sample)

t0, t1 = lin1.intercept_[0], lin1.coef_[0][0]
print(f"θ0={t0:.2f}, θ1={t1:.2e}")

In [ ]:
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

X = np.linspace(min_gdp, max_gdp, 1000)
plt.plot(X, t0 + t1 * X, "b")   # 학습된 최적 직선

plt.text(max_gdp - 20_000, min_life_sat + 1.9,
         fr"$\theta_0 = {t0:.2f}$", color="b")
plt.text(max_gdp - 20_000, min_life_sat + 1.3,
         fr"$\theta_1 = {t1 * 1e5:.2f} \times 10^{{-5}}$", color="b")

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()

In [ ]:
# 푸에르토리코 1인당 GDP
puerto_rico_gdp_per_capita = gdp_per_capita[gdppc_col].loc["Puerto Rico"]
puerto_rico_gdp_per_capita

In [ ]:
# 학습된 직선으로 예측
puerto_rico_predicted_life_satisfaction = lin1.predict(
    [[puerto_rico_gdp_per_capita]])[0, 0]
puerto_rico_predicted_life_satisfaction

In [ ]:
country_stats.plot(kind='scatter', figsize=(5, 3), grid=True,
                   x=gdppc_col, y=lifesat_col)

X = np.linspace(min_gdp, max_gdp, 1000)
plt.plot(X, t0 + t1 * X, "b")

plt.text(min_gdp + 22_000, max_life_sat - 1.1,
         fr"$\theta_0 = {t0:.2f}$", color="b")
plt.text(min_gdp + 22_000, max_life_sat - 0.6,
         fr"$\theta_1 = {t1 * 1e5:.2f} \times 10^{{-5}}$", color="b")

# 예측 위치를 빨간 점선과 점으로 표시
plt.plot([puerto_rico_gdp_per_capita, puerto_rico_gdp_per_capita],
         [min_life_sat, puerto_rico_predicted_life_satisfaction], "r--")
plt.text(puerto_rico_gdp_per_capita + 1000, 5.0,
         fr"Prediction = {puerto_rico_predicted_life_satisfaction:.2f}",
         color="r")
plt.plot(puerto_rico_gdp_per_capita, puerto_rico_predicted_life_satisfaction,
         "ro")

plt.axis([min_gdp, max_gdp, min_life_sat, max_life_sat])
plt.show()

## 3.5 샘플링 편향 (Nonrepresentative data)

잘라냈던 나라들을 다시 보여줍니다. 가난한 나라와 아주 부유한 나라(노르웨이, 아일랜드, 룩셈부르크)는 중간 구간에서 찾은 직선을 따르지 않습니다.

→ 비유: 서울 강남 주민만 조사해서 "한국인 평균 소득"을 추정하는 것과 같습니다.

In [ ]:
# 학습에 쓰지 않았던 나라들
missing_data = full_country_stats[(full_country_stats[gdppc_col] < min_gdp) |
                                  (full_country_stats[gdppc_col] > max_gdp)]
missing_data

In [ ]:
position_text_missing_countries = {
    "South Africa": (20_000, 4.2),
    "Colombia": (6_000, 8.2),
    "Brazil": (18_000, 7.8),
    "Mexico": (24_000, 7.4),
    "Chile": (30_000, 7.0),
    "Norway": (51_000, 6.2),
    "Switzerland": (62_000, 5.7),
    "Ireland": (81_000, 5.2),
    "Luxembourg": (92_000, 4.7),
}

In [ ]:
full_country_stats.plot(kind='scatter', figsize=(8, 3),
                        x=gdppc_col, y=lifesat_col, grid=True)

# 빠졌던 나라들을 빨간 네모로 표시
for country, pos_text in position_text_missing_countries.items():
    pos_data_x, pos_data_y = missing_data.loc[country]
    plt.annotate(country, xy=(pos_data_x, pos_data_y),
                 xytext=pos_text, fontsize=12,
                 arrowprops=dict(facecolor='black', width=0.5,
                                 shrink=0.08, headwidth=5))
    plt.plot(pos_data_x, pos_data_y, "rs")

# 점선: 부분 데이터로 학습한 직선
X = np.linspace(0, 115_000, 1000)
plt.plot(X, t0 + t1 * X, "b:")

# 실선: 전체 데이터로 학습한 직선
lin_reg_full = linear_model.LinearRegression()
Xfull = np.c_[full_country_stats[gdppc_col]]   # (N_full, 1)
yfull = np.c_[full_country_stats[lifesat_col]] # (N_full, 1)
lin_reg_full.fit(Xfull, yfull)

t0full, t1full = lin_reg_full.intercept_[0], lin_reg_full.coef_[0][0]
plt.plot(X, t0full + t1full * X, "k")

plt.axis([0, 115_000, min_life_sat, max_life_sat])
plt.show()

## 3.6 과대적합 (Overfitting)

$x, x^2, \dots, x^{10}$을 특성으로 쓰는 10차 다항 회귀는 모든 점을 억지로 지나가려 하면서 크게 출렁입니다. 훈련 데이터의 **잡음까지 외워버린** 상태입니다.

`StandardScaler`가 필요한 이유: GDP가 10만 단위라 $x^{10}$은 $10^{50}$에 가까운 값이 됩니다. 스케일을 맞추지 않으면 수치 계산이 불안정해집니다.

In [ ]:
from sklearn import preprocessing
from sklearn import pipeline

full_country_stats.plot(kind='scatter', figsize=(8, 3),
                        x=gdppc_col, y=lifesat_col, grid=True)

poly = preprocessing.PolynomialFeatures(degree=10, include_bias=False)  # (N,1) -> (N,10)
scaler = preprocessing.StandardScaler()                                # 평균 0, 표준편차 1
lin_reg2 = linear_model.LinearRegression()

# 다항 특성 생성 → 표준화 → 선형 회귀
pipeline_reg = pipeline.Pipeline([
    ('poly', poly),
    ('scal', scaler),
    ('lin', lin_reg2)])
pipeline_reg.fit(Xfull, yfull)

curve = pipeline_reg.predict(X[:, np.newaxis])   # X: (1000,) -> (1000, 1)
plt.plot(X, curve)

plt.axis([0, 115_000, min_life_sat, max_life_sat])
plt.show()

**우연한 패턴의 함정**: 이름에 `W`가 들어간 나라는 삶의 만족도가 모두 7점 이상입니다. 그럴듯한 규칙처럼 보이지만...

In [ ]:
w_countries = [c for c in full_country_stats.index if "W" in c.upper()]
full_country_stats.loc[w_countries][lifesat_col]

전체 GDP 데이터에서 `W`가 들어간 나라를 보면 Malawi, Rwanda, Zimbabwe처럼 GDP가 매우 낮은 나라도 많습니다. 우연한 패턴을 규칙으로 착각하는 것이 과대적합입니다.

In [ ]:
all_w_countries = [c for c in gdp_per_capita.index if "W" in c.upper()]
gdp_per_capita.loc[all_w_countries].sort_values(by=gdppc_col)

## 3.7 규제 (Regularization)

릿지 회귀의 비용 함수는 다음과 같습니다.

$$
J(\theta) = \underbrace{\sum_i (y_i - \hat{y}_i)^2}_{\text{예측 오차}} + \underbrace{\alpha\,\theta_1^2}_{\text{기울기 벌점}}
$$

- 첫째 항: 데이터에 얼마나 잘 맞는지
- 둘째 항: 모델이 얼마나 "과격한지"에 대한 벌금

부분 데이터로 학습해도 기울기가 완만해져서 전체 데이터 직선에 더 가까워집니다. `alpha`처럼 **모델이 아니라 학습 알고리즘을 조절하는 값**을 **하이퍼파라미터**라고 합니다.

In [ ]:
country_stats.plot(kind='scatter', x=gdppc_col, y=lifesat_col, figsize=(8, 3))
missing_data.plot(kind='scatter', x=gdppc_col, y=lifesat_col,
                  marker="s", color="r", grid=True, ax=plt.gca())

X = np.linspace(0, 115_000, 1000)
plt.plot(X, t0 + t1*X, "b:", label="Linear model on partial data")
plt.plot(X, t0full + t1full * X, "k-", label="Linear model on all data")

# alpha가 클수록 θ1을 0 쪽으로 강하게 끌어당김
ridge = linear_model.Ridge(alpha=10**9.5)
X_sample = country_stats[[gdppc_col]]
y_sample = country_stats[[lifesat_col]]
ridge.fit(X_sample, y_sample)
t0ridge, t1ridge = ridge.intercept_[0], ridge.coef_[0]
plt.plot(X, t0ridge + t1ridge * X, "b--",
         label="Regularized linear model on partial data")
plt.legend(loc="lower right")

plt.axis([0, 115_000, min_life_sat, max_life_sat])
plt.show()

# 4. 연습문제 해답 (Exercise Solutions)

1. **머신러닝이란?** 데이터로부터 학습하는 시스템을 만드는 것. 학습이란 어떤 성능 지표 기준으로 작업을 점점 더 잘하게 되는 것.
2. **어디에 유용한가?** 알고리즘으로 풀기 어려운 복잡한 문제, 긴 수작업 규칙 목록을 대체할 때, 변하는 환경에 적응해야 할 때, 사람이 데이터에서 통찰을 얻고 싶을 때(데이터 마이닝).
3. **레이블된 훈련 세트?** 각 샘플마다 정답(레이블)이 붙어 있는 데이터.
4. **대표적 지도 학습 작업?** 회귀와 분류.
5. **대표적 비지도 학습 작업?** 군집, 시각화, 차원 축소, 연관 규칙 학습.
6. **미지의 지형을 걷는 로봇?** 강화 학습이 가장 적합.
7. **고객 세그먼트 나누기?** 그룹 정의를 모르면 군집(비지도), 원하는 그룹이 정해져 있으면 분류(지도).
8. **스팸 감지는 지도/비지도?** 지도 학습 (스팸/정상 레이블이 있는 메일로 학습).
9. **온라인 학습?** 데이터를 조금씩 받아 점진적으로 학습. 변화하는 데이터에 빠르게 적응하고, 대용량 데이터도 학습 가능.
10. **외부 메모리(out-of-core) 학습?** 메모리에 다 안 들어가는 데이터를 미니배치로 쪼개 온라인 학습 방식으로 학습.
11. **사례 기반 학습?** 훈련 데이터를 외워두고, 새 샘플이 오면 유사도 측정으로 가장 비슷한 샘플을 찾아 예측 (예: k-NN).
12. **모델 파라미터 vs 하이퍼파라미터?** 파라미터는 모델의 예측을 결정하는 값(예: 기울기), 하이퍼파라미터는 학습 알고리즘의 설정값(예: 규제 강도).
13. **모델 기반 학습?** 비용 함수(예측 오차 + 규제 벌점)를 최소화하는 파라미터를 찾고, 새 샘플의 특성을 예측 함수에 넣어 예측.
14. **주요 도전 과제?** 데이터 부족, 낮은 품질, 대표성 없는 데이터, 무의미한 특성, 과소적합(너무 단순한 모델), 과대적합(너무 복잡한 모델).
15. **훈련에선 좋은데 새 데이터에선 나쁘다면?** 과대적합. 해결책: 데이터 추가, 모델 단순화(파라미터·특성 축소, 규제), 훈련 데이터의 잡음 제거.
16. **테스트 세트의 역할?** 실서비스 투입 전, 새 샘플에 대한 일반화 오차를 추정.
17. **검증 세트의 역할?** 여러 모델을 비교하고 하이퍼파라미터를 튜닝.
18. **train-dev 세트?** 훈련 데이터와 검증/테스트 데이터 분포가 다를 위험이 있을 때 사용. 훈련 세트 일부를 떼어 두고 평가.
    - 훈련 ◯, train-dev ✕ → 과대적합
    - 훈련 ◯, train-dev ◯, 검증 ✕ → 데이터 불일치 → 훈련 데이터를 실제 데이터와 비슷하게 개선
19. **테스트 세트로 하이퍼파라미터를 튜닝하면?** 테스트 세트에 과대적합되어 일반화 오차가 낙관적으로 측정됨 → 실서비스에서 기대보다 성능이 낮게 나올 수 있음.

# 📌 핵심 요약

- **학습 = 파라미터 찾기**: 선형 회귀는 $\theta_0, \theta_1$ 두 숫자로 직선을 결정하고, 예측은 그 식에 대입하는 것뿐입니다.
- **모델 기반 vs 사례 기반**: 선형 회귀는 규칙(식)을 배우고, k-NN은 데이터를 외워 이웃의 평균으로 예측합니다. 같은 데이터에서도 6.02 vs 5.73처럼 결과가 다릅니다.
- **대표성 없는 데이터**: 일부 구간만으로 학습하면 전체 경향과 다른 모델이 나옵니다.
- **과대적합**: 너무 복잡한 모델(10차 다항식)은 잡음과 우연한 패턴까지 외웁니다.
- **규제**: 비용 함수에 벌점 항을 더해 모델을 단순하게 유지하면 일반화가 좋아집니다. 규제 강도 `alpha`는 하이퍼파라미터입니다.
- **데이터 분할**: 훈련 세트로 학습, 검증 세트로 모델 선택·튜닝, 테스트 세트로 최종 일반화 성능 평가 — 테스트 세트는 마지막까지 손대지 않습니다.